In [12]:
!pip install wandb


In [36]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/icml_face_data.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/fer2013.tar.gz
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/example_submission.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv
/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/test.csv
/kaggle/input/notebooks/sandrolomidze42/model-py/__results__.html
/kaggle/input/notebooks/sandrolomidze42/model-py/__notebook__.ipynb
/kaggle/input/notebooks/sandrolomidze42/model-py/__output__.json
/kaggle/input/notebooks/sandrolomidze42/model-py/custom.css


In [19]:
!echo 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 > wandb login 

In [37]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense

def create_simple_cnn():
    """
    Minimal CNN: 1 Conv layer + ReLU -> Flatten -> Dense Output
    """
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(48, 48, 1)),
        Flatten(),
        Dense(7, activation='softmax')
    ])
    return model

In [38]:


model = create_simple_cnn()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 67712)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 7)              │       473,991 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 474,311 (1.81 MB)

 Trainable params: 474,311 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load training data
train_df = pd.read_csv('/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv')

# Convert pixel strings to arrays and reshape to 48x48 images
pixels = train_df['pixels'].apply(lambda x: np.array(x.split(), dtype=np.float32))
images = np.stack(pixels).reshape(-1, 48, 48, 1)

# Normalize pixel values to 
images = images / 255.0

# Labels (0-6)
labels = train_df['emotion'].values

# Split into training and validation sets (80% train, 20% val)
X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=0.2, random_state=42, stratify=labels)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")


Training set shape: (22967, 48, 48, 1)
Validation set shape: (5742, 48, 48, 1)


In [40]:
import wandb
import tensorflow as tf

# 1. Login
wandb.login(key="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")

# 2. Initialize the run
run = wandb.init(
    entity="slomi23-free-university-of-tbilisi-",
    project="fer2013-cnn",
    dir="/kaggle/working",
    mode="online", 
    config={
        "learning_rate": 0.001,
        "architecture": "SimpleCNN",
        "dataset": "FER2013",
        "epochs": 50,
        "batch_size": 32,
    }
)

print(f"W&B Run URL: {run.url}")

# 3. Use WandbMetricsLogger instead of WandbCallback
# This logs metrics (loss, accuracy) without trying to log the model graph
wandb_metrics_logger = wandb.keras.WandbMetricsLogger()

# 4. Define Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

# 5. Train the model
history = model.fit(
    X_train, y_train, 
    batch_size=32,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, wandb_metrics_logger]
)

# 6. Log the final model weights manually if needed
# wandb.save("model.h5") # You would need to save the model first

# 7. Finish the run
wandb.finish()


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


W&B Run URL: https://wandb.ai/slomi23-free-university-of-tbilisi-/fer2013-cnn/runs/4gu0y31w
Epoch 1/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.3546 - loss: 1.6722 - val_accuracy: 0.4216 - val_loss: 1.5343
Epoch 2/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.4676 - loss: 1.4256 - val_accuracy: 0.4350 - val_loss: 1.5077
Epoch 3/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5292 - loss: 1.2795 - val_accuracy: 0.4436 - val_loss: 1.4919
Epoch 4/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.5774 - loss: 1.1618 - val_accuracy: 0.4451 - val_loss: 1.5174
Epoch 5/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6170 - loss: 1.0556 - val_accuracy: 0.4504 - val_loss: 1.5278
Epoch 6/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6576 - loss: 0.9629 - val_accuracy: 0.4436 - val_loss: 1.5883
Epoch 7/50
718/718 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6917 - loss: 0.8745 - val_accuracy: 0.4486 - val_loss: 1.6459
Epoch 8/50
7

epoch/accuracy,▁▃▄▅▆▇▇█
epoch/epoch,▁▂▃▄▅▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁
epoch/loss,█▆▅▄▃▂▂▁
epoch/val_accuracy,▁▄▆▇█▆▇█
epoch/val_loss,▃▂▁▂▂▄▆█
epoch/accuracy,0.72696
epoch/epoch,7
epoch/learning_rate,0.001
epoch/loss,0.79313
epoch/val_accuracy,0.45106


In [41]:
val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc:.4f}")


180/180 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4436 - loss: 1.4919
Validation Loss: 1.4919
Validation Accuracy: 0.4436
